# General-Purpose LLM Therapist Benchmark (Gemini 2.5 Flash) — Drive version

This is a variant of `colab_drive_version.ipynb` for the next phase of the
study: instead of a **self-hosted domain-specific model** (MentaLLaMA,
ChatPsychiatrist) as the Therapist agent, the Therapist under test is now a
**general-purpose API model** — starting with **Gemini 2.5 Flash**.

Key differences from the self-hosted notebook:

- **No GPU needed.** Gemini is called over the API, so a CPU runtime is
  fine — you can skip `Runtime -> Change runtime type -> GPU` entirely.
- **No model loading / 4-bit quantization cell.** `THERAPIST_PROVIDER` is
  just `'gemini'` and `THERAPIST_MODEL` is `'gemini-2.5-flash'`.
- **Patient and Evaluator are deliberately kept on *different* backends
  from the Therapist** (Mistral / Groq) so the model being benchmarked
  isn't also role-playing the patient or grading its own transcript.
- Same Drive persistence as before: project lives in your Drive, logs
  (`summary_metrics.csv`, transcripts, evaluations, charts, why-reports)
  are written straight to Drive as each session finishes.

To benchmark a different general-purpose model later (Llama 3 via Groq,
GPT-4/GPT-4o via OpenAI, etc.), you only need to change `THERAPIST_PROVIDER`
/ `THERAPIST_MODEL` in section 3 below — everything else stays the same.

**Runtime setup:** the default CPU runtime is fine. `Runtime -> Change
runtime type -> None` (no accelerator needed).


In [1]:
# Lighter install than the self-hosted notebook — no torch/transformers/
# bitsandbytes needed since the Therapist model now runs over an API too.
!pip install -q google-genai mistralai groq python-dotenv pandas matplotlib

## 1. Mount Drive and set up the project (one-time upload, then it persists)

The first time you run this cell it will ask you to pick
`psych_llm_benchmark.zip` and will unzip it into your Drive. Every time after
that, it finds the project already there and skips straight to loading it —
no re-upload needed. If you already ran the self-hosted notebook before,
this will reuse that same Drive folder (and the same growing
`summary_metrics.csv`), so the new Gemini rows land alongside your existing
MentaLLaMA / ChatPsychiatrist results for direct comparison.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

# Change this if you want the project stored under a different Drive path/name
# Make sure this path exactly matches where your unzipped 'psych_llm_benchmark' folder is located in your Drive.
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/psych-llm-benchmark'

if not os.path.exists(DRIVE_PROJECT_DIR):
    print("Project not found in Drive yet. Upload psych_llm_benchmark.zip once:")
    from google.colab import files
    uploaded = files.upload()  # select psych_llm_benchmark.zip
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content/drive/MyDrive/')
    print("Unzipped into Drive. If the extracted folder name doesn't match "
          "DRIVE_PROJECT_DIR above, update the variable and re-run this cell.")
else:
    print("Found existing project already in Drive - reusing it, no upload needed.")

os.chdir(DRIVE_PROJECT_DIR)
import sys
sys.path.insert(0, DRIVE_PROJECT_DIR)
print('Project loaded at:', DRIVE_PROJECT_DIR)
print('Logs will be written directly to:', os.path.join(DRIVE_PROJECT_DIR, 'logs'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found existing project already in Drive - reusing it, no upload needed.
Project loaded at: /content/drive/MyDrive/psych-llm-benchmark
Logs will be written directly to: /content/drive/MyDrive/psych-llm-benchmark/logs


## 2. Set API keys

All three agents (Patient, Therapist, Evaluator) are API-based this run, so
set whichever keys you need below. Get free keys at:
- Gemini: https://aistudio.google.com/apikey  (needed — this powers the Therapist)
- Mistral: https://console.mistral.ai/api-keys  (needed for the Evaluator, and Patient if you use it)
- Groq: https://console.groq.com/keys  (optional — only if you route Patient through Groq/Llama 3 instead)
- OpenAI: only needed later if you add GPT-4/GPT-4o as another Therapist to compare

You'll need to re-enter these each new Colab session (Drive doesn't store
secrets for you).


In [3]:
import os
from getpass import getpass

os.environ['GEMINI_API_KEY'] = getpass('Gemini API key: ')
os.environ['GROQ_API_KEY'] = getpass('Groq API key (optional, press Enter to skip): ')
# os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key (optional): ')


Gemini API key: ··········
Groq API key (optional, press Enter to skip): ··········


In [4]:
from agents.llm_clients import get_client

# --- Therapist agent: the model under test for this run ---
# The model 'gemini-2.5-flash' is no longer available. Updating to 'gemini-flash-latest'.
therapist_llm = get_client('gemini', 'gemini-flash-latest')

# --- Patient agent (pick ONE - keep it different from the Therapist model) ---
# The user wants the patient model to act as a psychological therapist.
# Removing the system_prompt as GroqClient does not support it directly in initialization.
patient_system_prompt = "You are a psychological therapist."
patient_llm = get_client('groq', 'llama-3.1-8b-instant')

# --- Evaluator agent (pick ONE - keep it independent from the Therapist) ---
#evaluator_llm = get_client('mistral', 'mistral-large-latest')
evaluator_llm = get_client('groq', 'llama-3.3-70b-versatile')

print('Therapist:', therapist_llm.model_name)
print('Patient:  ', patient_llm.model_name)
print('Evaluator:', evaluator_llm.model_name)

Therapist: gemini-flash-latest
Patient:   llama-3.1-8b-instant
Evaluator: llama-3.3-70b-versatile


## 4. Run the benchmark

Pick which of the 9 conditions to run. Each session is 10 exchanges by
default — start with 1-2 conditions to sanity-check quality/latency before
running the full set.

Because the project lives in Drive, **each session's result is saved to
Drive the moment it finishes.** If Colab disconnects halfway through this
list, just re-run the setup cells above and re-run this cell with only the
conditions that didn't finish yet — you won't lose or duplicate the ones
that already completed (`summary_metrics.csv` appends, and each session gets
a unique ID). Since Gemini is an API call rather than a local GPU model,
sessions here should run noticeably faster than the self-hosted runs —
it's safe to queue up more conditions at once.


In [5]:
from analytics.exporter import AnalyticsExporter
from run_benchmark import run_single_session
from prompts.patient_profiles import PATIENT_PROFILES

conditions_to_run = ['OCD']  # edit as needed, or list(PATIENT_PROFILES.keys()) for all 9
exporter = AnalyticsExporter()

for condition in conditions_to_run:
    run_single_session(condition, patient_llm, therapist_llm, evaluator_llm,
                        max_turns=10, exporter=exporter)


[GeminiClient] error on attempt 1: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 42.19661957s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model

RuntimeError: GeminiClient failed after 3 retries: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 39.03934464s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 

## 5. Build charts + qualitative 'why' report for this model

This reads **all** logged sessions in `logs/` — including any earlier
MentaLLaMA/ChatPsychiatrist/other-model runs already in this Drive project
— so the charts and why-report below compare Gemini 2.5 Flash directly
against your other Therapist models, not just against itself.


In [ ]:
from analytics.aggregator import load_all_evaluations, load_full_records
from analytics.visualize import generate_all_charts
from analytics.why_analysis import WhyAnalyzer

df = load_all_evaluations()
chart_paths = generate_all_charts(df)
print(chart_paths)

records = load_full_records()
why_analyzer = WhyAnalyzer(llm=evaluator_llm)
report_paths = why_analyzer.analyze_all(records)
print(report_paths)


## 6. Your results are already in Drive - nothing to download

No zip/download step needed. Everything below already exists in your Drive
folder and will still be there next time you open this notebook:


In [ ]:
print('Transcripts:', os.path.join(DRIVE_PROJECT_DIR, 'logs', 'transcripts'))
print('Evaluations:', os.path.join(DRIVE_PROJECT_DIR, 'logs', 'evaluations'))
print('Summary CSV:', os.path.join(DRIVE_PROJECT_DIR, 'logs', 'summary', 'summary_metrics.csv'))
print('Charts:', os.path.join(DRIVE_PROJECT_DIR, 'logs', 'summary', 'figures'))
print('Why-reports:', os.path.join(DRIVE_PROJECT_DIR, 'logs', 'summary', 'why_reports'))


In [ ]:
import shutil
from google.colab import files
import os

# Define the directory to be zipped
logs_dir = os.path.join(DRIVE_PROJECT_DIR, 'logs')

# Define the output zip file name and path
output_filename = 'psych_llm_benchmark_logs'
output_zip_path = f'/tmp/{output_filename}'

# Create the zip archive
shutil.make_archive(output_zip_path, 'zip', logs_dir)

print(f'Zipping logs from: {logs_dir}')
print(f'Created zip file: {output_zip_path}.zip')

# Offer the zip file for download
files.download(f'{output_zip_path}.zip')